# 🥇 Gold Layer Analytics Engineering

The Gold layer represents the business intelligence and analytical intelligence layer of the Enterprise AI Lakehouse platform.

This layer transforms curated Silver semantic datasets into:

* executive KPIs
* operational intelligence
* business metrics
* analytical summaries
* AI reasoning datasets
* agent-ready intelligence views

The Gold layer is optimized for:

* enterprise analytics
* executive reporting
* AI copilots
* operational decision intelligence
* semantic business reasoning
* enterprise agent workflows

Unlike Silver datasets, Gold datasets are intentionally aggregated, business-focused, and optimized for enterprise intelligence consumption.


# ⚙️ Environment & Gold Configuration Initialization

In [0]:
# ==========================================
# Environment & Gold Configuration
# ==========================================

from pyspark.sql.functions import *
from pyspark.sql.types import *

# Enterprise Configuration
CONFIG = {

    "catalog": "spark_catalog",
    "schema": "uber_ai",

    "environment": "dev"
}

# Gold Layer Tables
GOLD_TABLES = {

    # City Intelligence
    "city_kpis_gold":
        f"{CONFIG['schema']}.city_kpis_gold",

    # Driver Intelligence
    "driver_kpis_gold":
        f"{CONFIG['schema']}.driver_kpis_gold",

    # Rider Intelligence
    "rider_kpis_gold":
        f"{CONFIG['schema']}.rider_kpis_gold",

    # Operational Intelligence
    "ride_operational_metrics_gold":
        f"{CONFIG['schema']}.ride_operational_metrics_gold",

    # Surge Intelligence
    "surge_analytics_gold":
        f"{CONFIG['schema']}.surge_analytics_gold",

    # Cancellation Intelligence
    "cancellation_analytics_gold":
        f"{CONFIG['schema']}.cancellation_analytics_gold",

    # Executive Intelligence
    "executive_summary_gold":
        f"{CONFIG['schema']}.executive_summary_gold"
}

print("✅ Gold Layer Configuration Initialized")

for table_name, table_path in GOLD_TABLES.items():
    print(f"{table_name} --> {table_path}")

# 🥈 Read Silver Layer Tables

In [0]:
# ==========================================
# Read Silver Layer Tables
# ==========================================

cities_silver_df = spark.table(
    f"{CONFIG['schema']}.cities_silver"
)

pricing_zones_silver_df = spark.table(
    f"{CONFIG['schema']}.pricing_zones_silver"
)

drivers_silver_df = spark.table(
    f"{CONFIG['schema']}.drivers_silver"
)

riders_silver_df = spark.table(
    f"{CONFIG['schema']}.riders_silver"
)

rides_silver_df = spark.table(
    f"{CONFIG['schema']}.rides_silver"
)

trip_events_silver_df = spark.table(
    f"{CONFIG['schema']}.trip_events_silver"
)

operational_documents_df = spark.table(
    f"{CONFIG['schema']}.operational_documents"
)

print("✅ Silver Layer Tables Loaded")

# 🌍 City KPI Gold Transformation

In [0]:
# ==========================================
# City KPI Gold Transformation
# ==========================================

city_kpis_gold_df = (

    rides_silver_df

    .groupBy(

        "city_id",
        "city_name",
        "country",
        "region",
        "city_tier"
    )

    .agg(

        # Ride Metrics
        count("ride_id").alias("total_rides"),

        # Revenue Metrics
        round(
            sum("fare_amount"),
            2
        ).alias("total_revenue"),

        round(
            avg("fare_amount"),
            2
        ).alias("avg_fare_amount"),

        # Distance Metrics
        round(
            avg("distance_km"),
            2
        ).alias("avg_distance_km"),

        # Duration Metrics
        round(
            avg("ride_duration_minutes"),
            2
        ).alias("avg_ride_duration_minutes"),

        # Driver Metrics
        countDistinct("driver_id").alias("active_drivers"),

        # Rider Metrics
        countDistinct("rider_id").alias("active_riders"),

        # Premium Ride Metrics
        round(

            (
                sum(
                    when(
                        col("ride_value_category") == "PREMIUM",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("premium_ride_percentage"),

        # Cancellation Metrics
        round(

            (
                sum(
                    when(
                        col("ride_status") == "CANCELLED",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("cancellation_percentage")
    )

    # Revenue Category
    .withColumn(

        "city_revenue_category",

        when(
            col("total_revenue") >= 5000000,
            "MEGA_CITY"
        )
        .when(
            col("total_revenue") >= 2000000,
            "HIGH_REVENUE"
        )
        .otherwise("STANDARD")
    )

    # AI Summary
    .withColumn(

        "city_summary",

        concat_ws(

            " ",

            lit("City"),
            col("city_name"),

            lit("generated total revenue of"),
            col("total_revenue"),

            lit("from"),
            col("total_rides"),

            lit("rides with average fare amount"),
            col("avg_fare_amount"),

            lit("and cancellation percentage"),
            col("cancellation_percentage")
        )
    )

    # Metadata
    .withColumn(
        "gold_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("GOLD")
    )
)

print("✅ City KPI Gold Transformation Completed")

# 🚘 Driver KPI Gold Transformation

In [0]:
# ==========================================
# Driver KPI Gold Transformation
# ==========================================

driver_kpis_gold_df = (

    rides_silver_df

    .groupBy(

        "driver_id",
        "driver_name",
        "vehicle_type",
        "rating_category"
    )

    .agg(

        # Ride Metrics
        count("ride_id").alias("total_rides"),

        # Revenue Metrics
        round(
            sum("fare_amount"),
            2
        ).alias("total_revenue_generated"),

        round(
            avg("fare_amount"),
            2
        ).alias("avg_fare_amount"),

        # Distance Metrics
        round(
            avg("distance_km"),
            2
        ).alias("avg_distance_km"),

        # Duration Metrics
        round(
            avg("ride_duration_minutes"),
            2
        ).alias("avg_ride_duration_minutes"),

        # Rider Metrics
        countDistinct("rider_id").alias("unique_riders_served"),

        # Premium Ride Metrics
        round(

            (
                sum(
                    when(
                        col("ride_value_category") == "PREMIUM",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("premium_ride_percentage"),

        # Cancellation Metrics
        round(

            (
                sum(
                    when(
                        col("ride_status") == "CANCELLED",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("cancellation_percentage")
    )

    # Driver Performance Tier
    .withColumn(

        "driver_performance_tier",

        when(
            col("total_revenue_generated") >= 100000,
            "ELITE"
        )
        .when(
            col("total_revenue_generated") >= 50000,
            "HIGH_PERFORMER"
        )
        .otherwise("STANDARD")
    )

    # AI Summary
    .withColumn(

        "driver_summary",

        concat_ws(

            " ",

            lit("Driver"),
            col("driver_name"),

            lit("completed"),
            col("total_rides"),

            lit("rides generating total revenue of"),
            col("total_revenue_generated"),

            lit("with average fare amount"),
            col("avg_fare_amount"),

            lit("and premium ride percentage"),
            col("premium_ride_percentage")
        )
    )

    # Metadata
    .withColumn(
        "gold_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("GOLD")
    )
)

print("✅ Driver KPI Gold Transformation Completed")

# 👤 Rider KPI Gold Transformation

In [0]:
# ==========================================
# Rider KPI Gold Transformation
# ==========================================

rider_kpis_gold_df = (

    rides_silver_df

    .groupBy(

        "rider_id",
        "rider_name",
        "rider_segment"
    )

    .agg(

        # Ride Metrics
        count("ride_id").alias("total_rides"),

        # Revenue Metrics
        round(
            sum("fare_amount"),
            2
        ).alias("total_spend"),

        round(
            avg("fare_amount"),
            2
        ).alias("avg_fare_amount"),

        # Distance Metrics
        round(
            avg("distance_km"),
            2
        ).alias("avg_distance_km"),

        # Duration Metrics
        round(
            avg("ride_duration_minutes"),
            2
        ).alias("avg_ride_duration_minutes"),

        # City Metrics
        countDistinct("city_id").alias("cities_travelled"),

        # Premium Ride Metrics
        round(

            (
                sum(
                    when(
                        col("ride_value_category") == "PREMIUM",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("premium_ride_percentage"),

        # Cancellation Metrics
        round(

            (
                sum(
                    when(
                        col("ride_status") == "CANCELLED",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("cancellation_percentage")
    )

    # Rider Value Tier
    .withColumn(

        "rider_value_tier",

        when(
            col("total_spend") >= 100000,
            "VIP"
        )
        .when(
            col("total_spend") >= 50000,
            "PREMIUM"
        )
        .otherwise("STANDARD")
    )

    # AI Summary
    .withColumn(

        "rider_summary",

        concat_ws(

            " ",

            lit("Rider"),
            col("rider_name"),

            lit("completed"),
            col("total_rides"),

            lit("rides with total spend of"),
            col("total_spend"),

            lit("across"),
            col("cities_travelled"),

            lit("cities and premium ride percentage"),
            col("premium_ride_percentage")
        )
    )

    # Metadata
    .withColumn(
        "gold_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("GOLD")
    )
)

print("✅ Rider KPI Gold Transformation Completed")

# 📊 Ride Operational Metrics Gold Transformation

In [0]:
# ==========================================
# Ride Operational Metrics Gold Transformation
# ==========================================

ride_operational_metrics_gold_df = (

    rides_silver_df

    .groupBy(

        "event_date",
        "city_name",
        "ride_time_category"

    )

    .agg(

        # Ride Metrics
        count("ride_id").alias("total_rides"),

        # Revenue Metrics
        round(
            sum("fare_amount"),
            2
        ).alias("total_revenue"),

        round(
            avg("fare_amount"),
            2
        ).alias("avg_fare_amount"),

        # Distance Metrics
        round(
            avg("distance_km"),
            2
        ).alias("avg_distance_km"),

        # Duration Metrics
        round(
            avg("ride_duration_minutes"),
            2
        ).alias("avg_ride_duration_minutes"),

        # Driver Metrics
        countDistinct("driver_id").alias("active_drivers"),

        # Rider Metrics
        countDistinct("rider_id").alias("active_riders"),

        # Completed Ride %
        round(

            (
                sum(
                    when(
                        col("ride_status") == "COMPLETED",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("completed_ride_percentage"),

        # Cancelled Ride %
        round(

            (
                sum(
                    when(
                        col("ride_status") == "CANCELLED",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("cancelled_ride_percentage"),

        # Premium Ride %
        round(

            (
                sum(
                    when(
                        col("ride_value_category") == "PREMIUM",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("premium_ride_percentage")
    )

    # Operational Health
    .withColumn(

        "operational_health_status",

        when(
            col("cancelled_ride_percentage") >= 20,
            "CRITICAL"
        )
        .when(
            col("cancelled_ride_percentage") >= 10,
            "WARNING"
        )
        .otherwise("STABLE")
    )

    # AI Summary
    .withColumn(

        "operational_summary",

        concat_ws(

            " ",

            lit("City"),
            col("city_name"),

            lit("recorded"),
            col("total_rides"),

            lit("rides generating revenue of"),
            col("total_revenue"),

            lit("with completed ride percentage"),
            col("completed_ride_percentage"),

            lit("and operational health status"),
            col("operational_health_status")
        )
    )

    # Metadata
    .withColumn(
        "gold_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("GOLD")
    )
)

print("✅ Ride Operational Metrics Gold Transformation Completed")

# 📈 Surge Analytics Gold Transformation

In [0]:
# ==========================================
# Surge Analytics Gold Transformation
# ==========================================

surge_analytics_gold_df = (

    rides_silver_df

    .groupBy(

        "city_name",
        "zone_name",
        "surge_category",
        "ride_time_category"

    )

    .agg(

        # Ride Metrics
        count("ride_id").alias("total_rides"),

        # Revenue Metrics
        round(
            sum("fare_amount"),
            2
        ).alias("total_revenue"),

        round(
            avg("fare_amount"),
            2
        ).alias("avg_fare_amount"),

        # Distance Metrics
        round(
            avg("distance_km"),
            2
        ).alias("avg_distance_km"),

        # Premium Ride %
        round(

            (
                sum(
                    when(
                        col("ride_value_category") == "PREMIUM",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("premium_ride_percentage"),

        # Completed Ride %
        round(

            (
                sum(
                    when(
                        col("ride_status") == "COMPLETED",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("completed_ride_percentage")
    )

    # Demand Intensity
    .withColumn(

        "demand_intensity",

        when(
            col("total_rides") >= 10000,
            "EXTREME_DEMAND"
        )
        .when(
            col("total_rides") >= 5000,
            "HIGH_DEMAND"
        )
        .when(
            col("total_rides") >= 2000,
            "MEDIUM_DEMAND"
        )
        .otherwise("LOW_DEMAND")
    )

    # Pricing Intelligence
    .withColumn(

        "pricing_efficiency",

        when(
            col("avg_fare_amount") >= 1500,
            "HIGH_VALUE"
        )
        .when(
            col("avg_fare_amount") >= 800,
            "MEDIUM_VALUE"
        )
        .otherwise("STANDARD_VALUE")
    )

    # AI Summary
    .withColumn(

        "surge_summary",

        concat_ws(

            " ",

            lit("Zone"),
            col("zone_name"),

            lit("in city"),
            col("city_name"),

            lit("recorded"),
            col("total_rides"),

            lit("rides under"),
            col("surge_category"),

            lit("surge category with average fare amount"),
            col("avg_fare_amount"),

            lit("and demand intensity"),
            col("demand_intensity")
        )
    )

    # Metadata
    .withColumn(
        "gold_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("GOLD")
    )
)

print("✅ Surge Analytics Gold Transformation Completed")

# 🚨 Cancellation Analytics Gold Transformation

In [0]:
# ==========================================
# Cancellation Analytics Gold Transformation
# ==========================================

cancellation_analytics_gold_df = (

    rides_silver_df

    .groupBy(

        "city_name",
        "zone_name",
        "ride_time_category",
        "vehicle_type"

    )

    .agg(

        # Total Ride Metrics
        count("ride_id").alias("total_rides"),

        # Cancelled Ride Metrics
        sum(

            when(
                col("ride_status") == "CANCELLED",
                1
            ).otherwise(0)

        ).alias("cancelled_rides"),

        # Completed Ride Metrics
        sum(

            when(
                col("ride_status") == "COMPLETED",
                1
            ).otherwise(0)

        ).alias("completed_rides"),

        # Revenue Impact
        round(
            sum("fare_amount"),
            2
        ).alias("total_revenue"),

        # Avg Fare
        round(
            avg("fare_amount"),
            2
        ).alias("avg_fare_amount"),

        # Avg Distance
        round(
            avg("distance_km"),
            2
        ).alias("avg_distance_km")
    )

    # Cancellation %
    .withColumn(

        "cancellation_percentage",

        round(

            (
                col("cancelled_rides")
                / col("total_rides")
            ) * 100,

            2
        )
    )

    # Operational Risk Level
    .withColumn(

        "operational_risk_level",

        when(
            col("cancellation_percentage") >= 30,
            "CRITICAL"
        )
        .when(
            col("cancellation_percentage") >= 20,
            "HIGH"
        )
        .when(
            col("cancellation_percentage") >= 10,
            "MEDIUM"
        )
        .otherwise("LOW")
    )

    # Operational Stability
    .withColumn(

        "operational_stability",

        when(
            col("completed_rides") >=
            col("cancelled_rides") * 10,
            "STABLE"
        )
        .otherwise("UNSTABLE")
    )

    # AI Summary
    .withColumn(

        "cancellation_summary",

        concat_ws(

            " ",

            lit("Zone"),
            col("zone_name"),

            lit("in city"),
            col("city_name"),

            lit("recorded cancellation percentage of"),
            col("cancellation_percentage"),

            lit("during"),
            col("ride_time_category"),

            lit("rides with operational risk level"),
            col("operational_risk_level")
        )
    )

    # Metadata
    .withColumn(
        "gold_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("GOLD")
    )
)

print("✅ Cancellation Analytics Gold Transformation Completed")

# 🧠 Executive Summary Gold Transformation

In [0]:
# ==========================================
# Executive Summary Gold Transformation
# ==========================================

executive_summary_gold_df = (

    rides_silver_df

    .groupBy(
        "event_date"
    )

    .agg(

        # Enterprise Ride Metrics
        count("ride_id").alias("total_rides"),

        # Enterprise Revenue Metrics
        round(
            sum("fare_amount"),
            2
        ).alias("total_revenue"),

        round(
            avg("fare_amount"),
            2
        ).alias("avg_fare_amount"),

        # Enterprise Distance Metrics
        round(
            avg("distance_km"),
            2
        ).alias("avg_distance_km"),

        # Enterprise Duration Metrics
        round(
            avg("ride_duration_minutes"),
            2
        ).alias("avg_ride_duration_minutes"),

        # Active Driver Metrics
        countDistinct("driver_id").alias("active_drivers"),

        # Active Rider Metrics
        countDistinct("rider_id").alias("active_riders"),

        # Active City Metrics
        countDistinct("city_id").alias("active_cities"),

        # Premium Ride %
        round(

            (
                sum(
                    when(
                        col("ride_value_category") == "PREMIUM",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("premium_ride_percentage"),

        # Cancellation %
        round(

            (
                sum(
                    when(
                        col("ride_status") == "CANCELLED",
                        1
                    ).otherwise(0)
                ) / count("ride_id")
            ) * 100,

            2

        ).alias("cancellation_percentage")
    )

    # Enterprise Health Status
    .withColumn(

        "enterprise_health_status",

        when(
            col("cancellation_percentage") >= 20,
            "CRITICAL"
        )
        .when(
            col("cancellation_percentage") >= 10,
            "WARNING"
        )
        .otherwise("HEALTHY")
    )

    # Enterprise Revenue Tier
    .withColumn(

        "enterprise_revenue_tier",

        when(
            col("total_revenue") >= 50000000,
            "ULTRA_HIGH"
        )
        .when(
            col("total_revenue") >= 20000000,
            "HIGH"
        )
        .otherwise("STANDARD")
    )

    # Executive AI Summary
    .withColumn(

        "executive_summary",

        concat_ws(

            " ",

            lit("Enterprise operations generated"),

            col("total_revenue"),

            lit("revenue from"),

            col("total_rides"),

            lit("rides across"),

            col("active_cities"),

            lit("cities with cancellation percentage of"),

            col("cancellation_percentage"),

            lit("and enterprise health status"),

            col("enterprise_health_status")
        )
    )

    # Metadata
    .withColumn(
        "gold_created_ts",
        current_timestamp()
    )

    .withColumn(
        "data_layer",
        lit("GOLD")
    )
)

print("✅ Executive Summary Gold Transformation Completed")

# 🧹 Gold Layer Deduplication

In [0]:
# ==========================================
# Gold Layer Deduplication
# ==========================================

city_kpis_gold_df = (
    city_kpis_gold_df
    .dropDuplicates(["city_id"])
)

driver_kpis_gold_df = (
    driver_kpis_gold_df
    .dropDuplicates(["driver_id"])
)

rider_kpis_gold_df = (
    rider_kpis_gold_df
    .dropDuplicates(["rider_id"])
)

ride_operational_metrics_gold_df = (
    ride_operational_metrics_gold_df
    .dropDuplicates([
        "event_date",
        "city_name",
        "ride_time_category"
    ])
)

surge_analytics_gold_df = (
    surge_analytics_gold_df
    .dropDuplicates([
        "city_name",
        "zone_name",
        "surge_category",
        "ride_time_category"
    ])
)

cancellation_analytics_gold_df = (
    cancellation_analytics_gold_df
    .dropDuplicates([
        "city_name",
        "zone_name",
        "ride_time_category",
        "vehicle_type"
    ])
)

executive_summary_gold_df = (
    executive_summary_gold_df
    .dropDuplicates(["event_date"])
)

print("✅ Gold Layer Deduplication Completed")

# 💾 Persist Gold Layer Tables

In [0]:
# ==========================================
# Persist Gold Layer Tables
# ==========================================

# ==========================================
# city_kpis_gold
# ==========================================

city_kpis_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        GOLD_TABLES["city_kpis_gold"]
    )

print("✅ city_kpis_gold persisted")

# ==========================================
# driver_kpis_gold
# ==========================================

driver_kpis_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        GOLD_TABLES["driver_kpis_gold"]
    )

print("✅ driver_kpis_gold persisted")

# ==========================================
# rider_kpis_gold
# ==========================================

rider_kpis_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        GOLD_TABLES["rider_kpis_gold"]
    )

print("✅ rider_kpis_gold persisted")

# ==========================================
# ride_operational_metrics_gold
# ==========================================

ride_operational_metrics_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .saveAsTable(
        GOLD_TABLES["ride_operational_metrics_gold"]
    )

print("✅ ride_operational_metrics_gold persisted")

# ==========================================
# surge_analytics_gold
# ==========================================

surge_analytics_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        GOLD_TABLES["surge_analytics_gold"]
    )

print("✅ surge_analytics_gold persisted")

# ==========================================
# cancellation_analytics_gold
# ==========================================

cancellation_analytics_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        GOLD_TABLES["cancellation_analytics_gold"]
    )

print("✅ cancellation_analytics_gold persisted")

# ==========================================
# executive_summary_gold
# ==========================================

executive_summary_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .saveAsTable(
        GOLD_TABLES["executive_summary_gold"]
    )

print("✅ executive_summary_gold persisted")

print("✅ ALL GOLD TABLES PERSISTED SUCCESSFULLY")

# ✅ Gold Layer Validation

In [0]:
# ==========================================
# Gold Layer Validation
# ==========================================

gold_validation = [

    ("city_kpis_gold",
     spark.table(
         GOLD_TABLES["city_kpis_gold"]
     ).count()),

    ("driver_kpis_gold",
     spark.table(
         GOLD_TABLES["driver_kpis_gold"]
     ).count()),

    ("rider_kpis_gold",
     spark.table(
         GOLD_TABLES["rider_kpis_gold"]
     ).count()),

    ("ride_operational_metrics_gold",
     spark.table(
         GOLD_TABLES["ride_operational_metrics_gold"]
     ).count()),

    ("surge_analytics_gold",
     spark.table(
         GOLD_TABLES["surge_analytics_gold"]
     ).count()),

    ("cancellation_analytics_gold",
     spark.table(
         GOLD_TABLES["cancellation_analytics_gold"]
     ).count()),

    ("executive_summary_gold",
     spark.table(
         GOLD_TABLES["executive_summary_gold"]
     ).count())
]

gold_validation_df = spark.createDataFrame(
    gold_validation,
    ["gold_table", "record_count"]
)

display(gold_validation_df)